In [1]:
# Install cmdstanpy and arviz modules (just one time)
# !pip install cmdstanpy
# !pip install arviz

In [2]:
# Check if cmdstanpy and arviz are installed
# !pip list | grep cmdstanpy
# !pip list | grep arviz

In [ ]:
# Install cmdstan -- Just for the first time!
# from cmdstanpy import install_cmdstan
# install_cmdstan()

e:\amirhossein\Polimi\052499_BAYSIAN STATISTICS\Progetto_Bayesian_Statistics\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CmdStan install directory: C:\Users\Mohsen\.cmdstan
Installing CmdStan version: 2.37.0
Download successful, file: C:\Users\Mohsen\AppData\Local\Temp\tmp2ns5mftv
Extracting distribution


11:04:33 - cmdstanpy - WARNING - CmdStan installation failed.
Command "make build" failed
Command: ['mingw32-make', 'build', '-j1']
failed with error [WinError 2] The system cannot find the file specified



Unpacked download as cmdstan-2.37.0
Building version cmdstan-2.37.0, may take several minutes, depending on your system.


False

In [7]:
# You can then check the cmdstan path through this commands
from cmdstanpy import cmdstan_path
cmdstan_path()

'C:\\Users\\Mohsen\\.cmdstan\\cmdstan-2.37.0'

In [8]:
# Create folder to store .stan files
import os
if not os.path.exists("./stan"):
    os.mkdir("./stan")

# Import modules
import numpy as np
import arviz as az
import seaborn as sns
import matplotlib.pyplot as plt

# Import functions
from cmdstanpy import CmdStanModel
from tensorflow_probability.substrates import numpy as tfp
tfd = tfp.distributions

# Stan is a probabilistic programming language 

It implements state of the art HMC algorithms.

Specifically, its default is the No-U-Turn Sampler by [Hoffman and Gelman (2014)](https://www.jmlr.org/papers/volume15/hoffman14a/hoffman14a.pdf) with some further improvements.

Given a target probability density function, the sampling is divided in

1. **Adaptation**: the algorithm learns the "optimal" stepsize $\varepsilon$, integration time $L$ and mass matrix $M$ of the momentum conditional distribution. The idea is to optimize these parameters so that the acceptance rate of the algorithm is 80%. This is done by first reaching the typical set, then learning $M$ and finally $L$ and $\varepsilon$.

2. **Burnin**: A number of iterations that will be discarded

3. Proper **sampling**


By default 1000 iterations of adaptation, 0 of burnin and 1000 of sampling (no thinning!)

## Stan can sample from my distributions!

In [9]:
normal_code = """
    data {
        int<lower=0> dim;
        matrix[dim, dim] cov_chol;
    }
    
    parameters {
        vector[dim] x;
    }
    
    model {
        vector[dim] mu = rep_vector(0, dim);
        x ~ multi_normal_cholesky(mu, cov_chol);
    }
"""

# Set Data
d = 2
sigma = 0.99 ** np.abs(np.vstack([np.arange(d)] *d) - np.vstack([np.arange(d)] *d).T)
sigma_chol = np.linalg.cholesky(sigma)

# Write model to file
stan_file = "./stan/multi_normal.stan"
with open(stan_file, "w") as f:
    print(normal_code, file=f)

# Compile stan model
stan_model = CmdStanModel(stan_file=stan_file)

11:18:41 - cmdstanpy - INFO - compiling stan file C:\Users\Mohsen\AppData\Local\Temp\tmpyxfd0349\tmp1lpzi07n.stan to exe file E:\amirhossein\Polimi\052499_BAYSIAN STATISTICS\Progetto_Bayesian_Statistics\src\jupyter\stan\multi_normal.exe


ValueError: Failed to compile Stan model 'E:\amirhossein\Polimi\052499_BAYSIAN STATISTICS\Progetto_Bayesian_Statistics\src\jupyter\stan\multi_normal.stan'. Console:

Command: ['mingw32-make', 'STANCFLAGS+=--filename-in-msg=multi_normal.stan', 'C:/Users/Mohsen/AppData/Local/Temp/tmpyxfd0349/tmp1lpzi07n.exe']
failed with error [WinError 2] The system cannot find the file specified



In [ ]:
import cmdstanpy

# Check CmdStan path
print(cmdstanpy.cmdstan_path())

# If it returns None or errors, install CmdStan:
cmdstanpy.install_cmdstan(verbose=True)

C:\Users\Mohsen\.cmdstan\cmdstan-2.37.0
CmdStan install directory: C:\Users\Mohsen\.cmdstan
Installing CmdStan version: 2.37.0
Download successful, file: C:\Users\Mohsen\AppData\Local\Temp\tmpsp9o65zu
Extracting distribution


10:21:54 - cmdstanpy - WARNING - CmdStan installation failed.
Command "make build" failed
Command: ['mingw32-make', 'build', '-j1']
failed with error [WinError 2] The system cannot find the file specified



Unpacked download as cmdstan-2.37.0
Building version cmdstan-2.37.0, may take several minutes, depending on your system.


False

In [ ]:
# Prepare data list for stan
normal_data = {
    "dim": 2,
    "cov_chol": sigma_chol
}

# Run the sampler
stan_fit = stan_model.sample(data=normal_data, chains=4, parallel_chains=4,iter_warmup=1000, iter_sampling=5000)

# Convert chain to arviz format
cmdstanpy_data = az.from_cmdstanpy(stan_fit)

In [ ]:
# Some plot using arviz
az.plot_trace(cmdstanpy_data)
plt.show()

In [ ]:
az.plot_trace(cmdstanpy_data, compact=False)
plt.tight_layout()
plt.show()

Let'see how `stan` has explored the level set of the target distribution:

In [ ]:
# Plot the contour of the target distribution (i.e. 2D Gaussian distribution)
x = y = np.linspace(-2.5, 2.5, 200)
X, Y = np.meshgrid(x, y)
pos = np.hstack([X.reshape(-1, 1), Y.reshape(-1, 1)])
z = tfd.MultivariateNormalFullCovariance(np.zeros(2), sigma).prob(pos)
plt.contour(X, Y, z.reshape(X.shape))

# Add first N saved iterations
N = 100
plt.scatter(cmdstanpy_data.posterior.x[0, :N, 0], cmdstanpy_data.posterior.x[0, :N, 1], color="red")
plt.show()

### Compute running average of $\sum x_i$ when $d=10$

In [ ]:
d = 10
sigma = 0.8 ** np.abs(np.vstack([np.arange(d)] * d) - np.vstack([np.arange(d)] * d).T)
sigma_chol = np.linalg.cholesky(sigma)

normal_data = {
    "dim": d,
    "cov_chol": sigma_chol
}

stan_fit = stan_model.sample(data=normal_data, chains=4, parallel_chains=4, 
                             iter_warmup=1000, iter_sampling=5000, show_progress="notebook")

cmdstanpy_data = az.from_cmdstanpy(stan_fit)

In [ ]:
running_sum = np.cumsum(np.sum(cmdstanpy_data.posterior.x, axis=-1), axis=1)
running_avg = running_sum / np.arange(running_sum.shape[1])

for i in range(4):
    plt.plot(running_avg[i, :], label="Chain: {0}".format(i+1))
plt.legend(fontsize=16)
plt.show()

## Multiple chains are useful for assessing convergence - $\hat{R}$ and ESS

$\hat{R}$ is a diagnostic of convergence: Implementation in Stan follows [Vehtari et al. (2021)][1]. Check it out for further details.

In the equations below, $N$ is the number of draws per chain, $M$ is the number of chains, $S=MN$ is the total number of draws from all
chains, $\theta^{(nm)}$ is $n$-th draw of $m$-th chain, $\bar{\theta}^{(\bullet m)}$ is the average of draws from $m$-th chain, and $\bar{\theta}^{(\bullet\bullet)}$ is average of all draws.  

1. For each scalar quantity of interest $\theta$, first compute $B$ and $W$, the between- and within-chain variances:
\begin{align*}
   B &= \frac{N}{M-1} \sum_{m=1}^{M} \left( \bar{\theta}^{(\bullet m)} - \bar{\theta}^{(\bullet\bullet)} \right)^2 \\
   W &= \frac{1}{M} \sum_{m=1}^{M} s^{2}_{m} \quad\text{where}\quad s^{2}_{m} = \frac{1}{N-1}\sum_{n=1}^{N} \left( \theta^{(nm)} - \bar{\theta}^{(\bullet m)} \right)^2
\end{align*}
2. Estimate $\text{var}(\theta \mid y)$, the marginal posterior variance of the estimand, by a weighted average of $W$ and $B$:
\begin{equation*}
    \hat{\text{var}}^{+}\left( \theta \mid y \right) = \frac{N-1}{N}W + \frac{1}{N}B
\end{equation*}
3. Compute $\hat{R}$ as:
\begin{equation*}
    \hat{R} = \sqrt{\frac{\hat{\text{var}}^{+}\left( \theta \mid y \right)}{W}}
\end{equation*}

In practice, we monitor convergence of the iterative simulations to the target distribution by estimating the factor by which the scale of the current distribution for $\theta$ might be reduced if the simulations were continued in the limit $N \to \infty$. In fact, the above estimator, for an ergodic process, declines to $1$ as $N \to \infty$. Hence, a value as close as possible to $1$ means that the chain converged. **Rule of thumb**: if $\hat{R} > 1.1$, the chain did not converge!

> *Indeed the actual implementation of $\hat{R}$ in Stan does not use actual draws $\theta^{(nm)}$'s, but either their rank normalization or the rank normalization of the folded draws, i.e. the absolute deviation of $\theta^{(nm)}$ from the median. See [Vehtari et al. (2021)][1] for all the details.*

The **effective sample size** (ESS) is an estimate of the sample size required to achieve the same level of precision if that sample was a simple random sample. Still, refer to Section 3.2 in [Vehtari et al. (2021)][1] for the estimation of ESS with multiple chains. We want it to be as close as possible to the actual draws of our chain. Notice that with Stan, if we are close to normality, we can reach values that are higher than the sample size in case of parameters whose posterior is close to a Gaussian distribution and low dependence on other parameters. Long story short, this happens due to the fact that HMC can exhibt autocorrelation plots with negative autocorrelations on odd lags. In the paper, you can find more details about the computation of the so-called *bulk-ESS* and *tail-ESS*, both available in Stan.

[1]: https://projecteuclid.org/journalArticle/Download?urlId=10.1214%2F20-BA1221

In [ ]:
stan_fit.summary()

In [ ]:
az.ess(cmdstanpy_data, method="bulk").x

In [ ]:
az.ess(cmdstanpy_data, method="tail").x

In [ ]:
az.rhat(cmdstanpy_data).x

# Using stan to sample from Bayesian models

Just as before, but now the covariance is random, while x is observed

Model

\begin{equation*}
    \begin{aligned}
        x_1, \ldots, x_N \mid \Sigma & \sim \mathcal{N}(0, \Sigma) \\
        \Sigma & \sim \mathcal{IW}(\nu_0, \psi_0)
    \end{aligned}
\end{equation*}

In [ ]:
normal_unk_cov_code = """
    data {
        int<lower=0> dim;
        int<lower=0> N;
        array[N] vector[dim] x;
        
        real<lower=0> nu0;
        cov_matrix[dim] psi0;
    }
    
    parameters {
        cov_matrix[dim] sigma;
    }
    
    transformed parameters {
        matrix[dim, dim] sigma_chol = cholesky_decompose(sigma);
    }
    
    model {
        vector[dim] mu = rep_vector(0, dim);
        sigma ~ inv_wishart(nu0, psi0);
        x ~ multi_normal_cholesky(mu, sigma_chol);
    }
"""

# Generate data
d = 10
N = 500
sigma = 0.8 ** np.abs(np.vstack([np.arange(d)] *d) - np.vstack([np.arange(d)] *d).T)
sigma_chol = np.linalg.cholesky(sigma)
x = tfd.MultivariateNormalTriL(np.zeros(d), sigma_chol).sample(N)

# Write model to file
stan_file = "./stan/multi_normal_unk_cov.stan"
with open(stan_file, "w") as f:
    print(normal_unk_cov_code, file=f)

# Compile stan model
stan_model = CmdStanModel(stan_file=stan_file)

In [ ]:
# Prepare input list
normal_data = {
    "dim": d,
    "N": x.shape[0],
    "nu0": 15,
    "x": x,
    "psi0": (15 - d - 1) * np.eye(d)
}

# Sample
stan_fit = stan_model.sample(data=normal_data, chains=4, parallel_chains=4, 
                             iter_warmup=1000, iter_sampling=5000)

# Convert to arviz data type
cmdstanpy_data = az.from_cmdstanpy(stan_fit)

In [ ]:
az.rhat(cmdstanpy_data).sigma

In [ ]:
az.plot_trace(cmdstanpy_data, compact=False)
plt.tight_layout()
plt.show()

In [ ]:
np.sum(cmdstanpy_data.sample_stats.diverging)

## Going hierarchical

We consider 341 frogs from 12 genuses. For each frog we measure a function of the Body Weight (BoW) and the Brain Weight (BrW):

$$
    y_j = \log \left(\frac{BrW/BoW}{1 - BrW / BoW} \right) = \text{logit}(BrW/BoW), \quad j=1, \ldots, 341
$$

We want to study differences across the genuses


Denote with $g(j) \in \{1, \ldots, 12\}$ the genus of individual $j$. 
A simple hierarchical model is

\begin{equation*}
    \begin{aligned}
        y_j \mid \theta_1, \ldots, \theta_{12}, \sigma & \sim \mathcal{N}(\theta_{g(j)}, \sigma^2) \\
        \theta_1, \ldots, \theta_{12} \mid \mu, \tau &\sim \mathcal{N}(\mu, \tau^2) \\
        \mu, \tau & \sim \mathcal{N}(\mu_0, s_0^2) \times \text{Uniform}(0, \tau_{max}) \\
        \sigma & \sim \text{Uniform}(0, \sigma_{max})
    \end{aligned}
\end{equation*}

## How to set the prior?
The ratio between bodyweight and brainweight is well studied and also has a [Wikipedia page][1]. Since we are not biologist, we use this page to elicit our prior.

[1]: https://en.wikipedia.org/wiki/Brain%E2%80%93body_mass_ratio

In [ ]:
def invlogit(x):
    return np.exp(x) / (1 + np.exp(x))

def logit(x):
    return np.log(x) - np.log(1 - x)

In [ ]:
# Average brain to body ratio for frogs is 1/172
mu0 = logit(1 / 172)
print("mu0: {0}".format(mu0))

In [ ]:
from scipy.optimize import minimize

# P(a <= X <= b), with X ~ N(mu,s)
def p_in_interval(mu, s, a, b):
    dist = tfd.Normal(mu, s)
    return dist.cdf(b) - dist.cdf(a)

# Compute s0 s.t. P(a <= X <= b) >= 0.9
loss_fn = lambda x: (p_in_interval(mu0, x, logit(1.0 / 550), logit(1.0/125)) - 0.9) ** 2
s0 = minimize(loss_fn, 10).x[0]
print("s0: {0}".format(s0))

In [ ]:
# Raughly estimate the standard deviations of the ratios to estimate taumax
ratios = 1.0 / np.array([12.0, 40.0, 40, 100, 125, 172, 550, 560, 600, 2496, 2798])
logit_ratios = logit(ratios)
taumax = 5 * np.std(logit_ratios)
print("taumax: {0}".format(taumax))

In [ ]:
# No idea on sigmamax: set to 100
sigmamax = 100.0
print("sigmamax: {0}".format(sigmamax))

## Let's import the data

In [ ]:
import pandas as pd

frog_data = pd.read_csv("frog_data.csv")
frog_data.head()

In [ ]:
frog_y = logit(frog_data.BrW.values / frog_data.BoW.values)
ind2genus = np.where(
    frog_data.Genus.values[:, np.newaxis] == np.unique(frog_data.Genus))[1]

In [ ]:
fig = plt.figure()

for i, g in enumerate(np.unique(frog_data.Genus)):
    curr_y = frog_y[ind2genus == i]
    sns.kdeplot(curr_y)
    
plt.show()

In [ ]:
frogs_code = """
data {
    int<lower=0> J;
    int<lower=0> N;
    array[N] real y;
    array[N] int<lower=1, upper=J> g;

    real<lower=0> s0;
    real<lower=0> taumax;
    real mu0;
}

parameters {
    real mu;
    real<lower=0> tau;
    
    array[J] real theta;
    real<lower=0> sigma;
}


model {
    mu ~ normal(mu0, s0);
    tau ~ uniform(0, taumax);
    sigma ~ uniform(0, 100);
    theta ~ normal(mu, tau);
    for (i in 1:N) {
        y ~ normal(theta[g[i]], sigma);
    }
}
"""

# Write model to file
stan_file = "./stan/frogs_code.stan"
with open(stan_file, "w") as f:
    print(frogs_code, file=f)

# Compile stan Model
stan_model = CmdStanModel(stan_file=stan_file)

In [ ]:
# Prepare input list
frogs_data = {
    "J": 12,
    "N": len(frog_y),
    "y": frog_y,
    "g": ind2genus + 1,
    "s0": s0,
    "taumax": taumax,
    "mu0": mu0
}

# Sample
stan_fit = stan_model.sample(data=frogs_data)

# Convert to arviz data type
cmdstanpy_data = az.from_cmdstanpy(stan_fit)

In [ ]:
az.plot_trace(cmdstanpy_data, compact=False)
plt.tight_layout()

In [ ]:
num_div = int(np.sum(cmdstanpy_data.sample_stats.diverging))
print("We have {0} diverging iterations!".format(num_div))

In [ ]:
def split_diverging(data, var_name):
    div_iters = np.where(data.sample_stats.diverging)
    non_div_iters = np.where(data.sample_stats.diverging == False)
    vals = data.posterior[var_name].values
    return (vals[div_iters], vals[non_div_iters])

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(15, 5))

var_names = ["mu", "sigma", "tau"]
cols = np.array(["steelblue", "forestgreen"])

idx = 0
for i, v1 in enumerate(var_names):
    div1, nondiv1 = split_diverging(cmdstanpy_data, v1)
    for v2 in var_names[i+1:]:
        div2, nondiv2 = split_diverging(cmdstanpy_data, v2)
        axes[idx].scatter(nondiv1, nondiv2, color="steelblue", alpha=0.1)
        axes[idx].scatter(div1, div2, s=70, color="green")
        axes[idx].set_title("{0} vs {1}".format(v1, v2), fontsize=14)
        
        if v1 in ["sigma", "tau"]:
            axes[idx].set_xscale("log")
        
        if v2 in ["sigma", "tau"]:
            axes[idx].set_yscale("log")
            
        idx += 1

In [ ]:
fig, axes = plt.subplots(nrows=12, ncols=3, figsize=(10, 30))

var_names = ["mu", "sigma", "tau"]
cols = np.array(["steelblue", "forestgreen"])

divtheta, nondivtheta = split_diverging(cmdstanpy_data, "theta")

for i, v1 in enumerate(var_names):
    div1, nondiv1 = split_diverging(cmdstanpy_data, v1)        
    for j in range(12):
        axes[j][i].scatter(nondivtheta[:, j], nondiv1, color="steelblue", alpha=0.1)
        axes[j][i].scatter(divtheta[:, j], div1, color="forestgreen")
        axes[j][i].set_title("theta_{1} vs {0}".format(v1, j), fontsize=14)
        if v1 in ["sigma", "tau"]:
            axes[j][i].set_yscale("log")


plt.tight_layout()
plt.show()

In [ ]:
frogs_code_reparam = """
data {
    int<lower=0> J;
    int<lower=0> N;
    array[N] real y;
    array[N] int<lower=1, upper=J> g;
    
    real<lower=0> s0;
    real<lower=0> taumax;
    real mu0;
}

parameters {
    real mu;
    real<lower=0> tau;
    
    vector[J] theta_raw;
    real<lower=0> sigma;
}

transformed parameters {
    vector[J] theta;
    theta = mu + theta_raw * tau;
}


model {
    mu ~ normal(mu0, s0);
    tau ~ uniform(0, taumax);
    sigma ~ uniform(0, 100);
    theta_raw ~ normal(0, 1);
    for (i in 1:N) {
        y ~ normal(theta[g[i]], sigma);
    }
}
"""

stan_file = "./stan/frogs_code_reparam.stan"
with open(stan_file, "w") as f:
    print(frogs_code_reparam, file=f)
stan_model = CmdStanModel(stan_file=stan_file)

In [ ]:
stan_fit = stan_model.sample(data=frogs_data)
cmdstanpy_data = az.from_cmdstanpy(stan_fit)
np.sum(cmdstanpy_data.sample_stats.diverging)

In [ ]:
az.plot_trace(cmdstanpy_data, compact=False)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(15, 5))

var_names = ["mu", "sigma", "tau"]
cols = np.array(["steelblue", "forestgreen"])

idx = 0
for i, v1 in enumerate(var_names):
    div1, nondiv1 = split_diverging(cmdstanpy_data, v1)
    for v2 in var_names[i+1:]:
        div2, nondiv2 = split_diverging(cmdstanpy_data, v2)
        axes[idx].scatter(nondiv1, nondiv2, color="steelblue", alpha=0.4)
        axes[idx].scatter(div1, div2, color="forestgreen")
        axes[idx].set_title("{0} vs {1}".format(v1, v2), fontsize=14)
        
        if v1 in ["sigma", "tau"]:
            axes[idx].set_xscale("log")
        
        if v2 in ["sigma", "tau"]:
            axes[idx].set_yscale("log")
            
        idx += 1

In [ ]:
fig = plt.figure()

for i in range(12):
    sns.kdeplot(cmdstanpy_data.posterior.theta[:, i, :].values.reshape(-1, ))
    
plt.show()

# Other startegies to fix diverging iterations


Increase `adapt_delta` (target acceptance rate) from the default value (0.8) to (0.9, 0.95, 0.99, 0.999)

Decrease the initial `step_size` from 1 (default value)

Increse `max_treedepth` (something very specific of NUTS) from 10

In [ ]:
stan_fit = stan_model.sample(data=frogs_data, adapt_delta=0.99, 
                             step_size=0.5, max_treedepth=15)
cmdstanpy_data = az.from_cmdstanpy(stan_fit)
np.sum(cmdstanpy_data.sample_stats.diverging)

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(15, 5))

var_names = ["mu", "sigma", "tau"]
cols = np.array(["steelblue", "forestgreen"])

idx = 0
for i, v1 in enumerate(var_names):
    div1, nondiv1 = split_diverging(cmdstanpy_data, v1)
    for v2 in var_names[i+1:]:
        div2, nondiv2 = split_diverging(cmdstanpy_data, v2)
        axes[idx].scatter(nondiv1, nondiv2, color="steelblue", alpha=0.4)
        axes[idx].scatter(div1, div2, color="forestgreen")
        axes[idx].set_title("{0} vs {1}".format(v1, v2), fontsize=14)
        
        if v1 in ["sigma", "tau"]:
            axes[idx].set_xscale("log")
        
        if v2 in ["sigma", "tau"]:
            axes[idx].set_yscale("log")
            
        idx += 1